## STEP 0 — Stop vLLM (clean slate)

In [1]:
!fuser -k 8000/tcp || true
!pkill -f vllm || true

^C


In [2]:
!ss -ltnp | grep ':8000' || echo "vLLM stopped"

vLLM stopped


## STEP 1 - Install SGLang

In [3]:
!pip install -U sglang

## STEP 2 - Start SGLang server

In [4]:
!nohup python -m sglang.launch_server \
  --model gpt2 \
  --host 0.0.0.0 \
  --port 30000 > sglang_gpt2.log 2>&1 &

## STEP 3 - Verify SGLang is running

### Check Logs

In [5]:
!tail -n 40 sglang_gpt2.log

W0000 00:00:1765840957.828243    3825 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765840957.828245    3825 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765840957.828247    3825 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has

### Health check

In [7]:
!curl --max-time 5 -s -o /dev/null -w "%{http_code}\n" http://127.0.0.1:30000/health

200


## STEP 4 - Start GPU utilization logging

In [ ]:
# Use higher frequency (100ms) to catch fast GPU spikes for small models like GPT-2
!nohup bash -lc 'nvidia-smi --query-gpu=timestamp,utilization.gpu,utilization.memory,memory.used --format=csv -l 0.1 > gpu_util_sglang.csv' >/dev/null 2>&1 &
!echo "GPU logging started for SGLang (100ms sampling frequency)"

GPU logging started for SGLang


## Step 5 - Run SGLang Benchmark

In [ ]:
import time, requests, pandas as pd

URL = "http://127.0.0.1:30000/v1/completions"
MODEL = "gpt2"

PROMPTS = [
    "Explain cloud computing in one sentence.",
    "Summarize: model serving platforms need to balance latency, throughput, and cost.",
    "Write 3 bullet points comparing CPU vs GPU for inference.",
    "Explain what batching means in inference and why it improves throughput.",
]

def run_once(prompt, max_tokens=128, temperature=0.0):
    payload = {
        "model": MODEL,
        "prompt": prompt,
        "max_tokens": max_tokens,
        "temperature": temperature,
    }
    t0 = time.time()
    r = requests.post(URL, json=payload, timeout=180)
    t1 = time.time()
    r.raise_for_status()
    data = r.json()
    text = data["choices"][0]["text"]
    usage = data.get("usage", {})
    return {
        "latency_s": t1 - t0,
        "output_text": text,
        "prompt_tokens": usage.get("prompt_tokens", None),
        "completion_tokens": usage.get("completion_tokens", None),
    }

rows = []

# Warm-up
print("Running warm-up request...")
_ = run_once("Say hello.", max_tokens=32)
print("Warm-up complete. Starting benchmark...")

# Run benchmark with concurrent requests to keep GPU busy
# This helps capture GPU utilization for fast models like GPT-2
import concurrent.futures

def run_benchmark_trial(prompt, max_toks, trial):
    """Run a single benchmark trial"""
    out = run_once(prompt, max_tokens=max_toks)
    approx_tokens = len(out["output_text"].split())
    gen_tokens = out["completion_tokens"] if out["completion_tokens"] else approx_tokens
    return {
        "framework": "SGLang",
        "model": MODEL,
        "max_tokens": max_toks,
        "prompt_len_chars": len(prompt),
        "trial": trial,
        "latency_s": out["latency_s"],
        "gen_tokens": gen_tokens,
        "tokens_per_sec": gen_tokens / out["latency_s"] if out["latency_s"] > 0 else None,
    }

# Run requests sequentially (as before) but with better timing
for max_toks in [64, 128, 256]:
    print(f"Running benchmark for max_tokens={max_toks}...")
    for prompt in PROMPTS:
        for trial in range(5):
            result = run_benchmark_trial(prompt, max_toks, trial)
            rows.append(result)
    print(f"Completed max_tokens={max_toks}")

print(f"Benchmark complete! Total trials: {len(rows)}")

df_sglang_gpt2 = pd.DataFrame(rows)
df_sglang_gpt2.head()

,framework,model,max_tokens,prompt_len_chars,trial,latency_s,gen_tokens,tokens_per_sec
0,SGLang,gpt2,64,40,0,0.110213,64,580.693530
1,SGLang,gpt2,64,40,1,8.205664,64,7.799491
2,SGLang,gpt2,64,40,2,0.110749,64,577.884533
3,SGLang,gpt2,64,40,3,0.109981,64,581.918377
4,SGLang,gpt2,64,40,4,0.109216,64,585.993582


## Step 6 - Stop GPU logging

In [10]:
!pkill -f "gpu_util_sglang.csv" || true
!echo "GPU logging stopped"

^C
GPU logging stopped


## Step 7 - Save SGLang results

In [11]:
df_sglang_gpt2.to_csv("sglang_gpt2_results.csv", index=False)
print("Saved sglang_gpt2_results.csv")

Saved sglang_gpt2_results.csv


In [ ]:
# Use higher frequency (100ms) for better GPU utilization capture
!nohup bash -lc 'nvidia-smi --query-gpu=timestamp,utilization.gpu,utilization.memory,memory.used --format=csv -l 0.1 > gpu_util_sglang_gpt2.csv' >/dev/null 2>&1 &

## STEP 8 - Quick sanity checks

In [13]:
!head gpu_util_sglang.csv
!tail gpu_util_sglang.csv

timestamp, utilization.gpu [%], utilization.memory [%], memory.used [MiB]
2025/12/15 23:23:40.931, 0 %, 0 %, 73271 MiB
2025/12/15 23:23:41.932, 0 %, 0 %, 73271 MiB
2025/12/15 23:23:42.933, 0 %, 0 %, 73271 MiB
2025/12/15 23:23:43.933, 0 %, 0 %, 73271 MiB
2025/12/15 23:23:44.934, 0 %, 0 %, 73271 MiB
2025/12/15 23:23:45.935, 0 %, 0 %, 73271 MiB
2025/12/15 23:23:46.936, 0 %, 0 %, 73271 MiB
2025/12/15 23:23:47.938, 0 %, 0 %, 73271 MiB
2025/12/15 23:23:48.939, 0 %, 0 %, 73271 MiB
2025/12/15 23:24:20.967, 0 %, 0 %, 73271 MiB
2025/12/15 23:24:21.967, 0 %, 0 %, 73271 MiB
2025/12/15 23:24:22.968, 0 %, 0 %, 73271 MiB
2025/12/15 23:24:23.969, 0 %, 0 %, 73271 MiB
2025/12/15 23:24:24.970, 0 %, 0 %, 73271 MiB
2025/12/15 23:24:25.971, 0 %, 0 %, 73271 MiB
2025/12/15 23:24:26.972, 0 %, 0 %, 73271 MiB
2025/12/15 23:24:27.972, 0 %, 0 %, 73271 MiB
2025/12/15 23:24:28.973, 0 %, 0 %, 73271 MiB
2025/12/15 23:24:29.974, 0 %, 0 %, 73271 MiB
